# Photonic Qubit Simulation

This notebook models a paraxial Gaussian beam, an ideal thin lens, and a qubit encoded in horizontal and vertical polarization. The optical parameters below are illustrative.

We use the normalized spatial intensity and track the lens with the complex beam parameter (q). The Jones vector represents the polarization and is assumed uniform across the beam. In this model the joint electric field is a scalar Gaussian envelope multiplied by the Jones vector. We evaluate the beam intensity without computing its spatial phase.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

figure_dir = Path("figures") if Path("notebooks").is_dir() else Path("../figures")
figure_dir.mkdir(parents=True, exist_ok=True)

wavelength = 632.8e-9  # m
waist = 0.40e-3       # m
rayleigh = np.pi * waist**2 / wavelength
lens_position = 0.30  # m from the initial waist
focal_length = 0.20   # m

plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.labelcolor": "#223447", "text.color": "#223447",
    "font.size": 11, "figure.dpi": 120,
})


def save_figure(fig, name):
    fig.tight_layout()
    fig.savefig(figure_dir / name, dpi=180, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)

## Gaussian beam propagation

In free space (q(z)=z+i z_R), with (z_R=\pi w_0^2/\lambda). The beam radius is (w(z)=w_0\sqrt{1+(z/z_R)^2}). At that radius the intensity falls to (e^{-2}) of the local on-axis intensity. The cross-section below uses a common normalization, (I(0,0)=1), so the color also shows how the peak intensity decreases as the beam spreads.

In [ ]:
def beam_radius(q):
    return np.sqrt(-wavelength / (np.pi * np.imag(1 / q)))


z = np.linspace(-2 * rayleigh, 2 * rayleigh, 450)
radius = beam_radius(z + 1j * rayleigh)
np.testing.assert_allclose(radius, waist * np.sqrt(1 + (z / rayleigh)**2))
np.testing.assert_allclose(beam_radius(rayleigh + 1j * rayleigh), np.sqrt(2) * waist)

x = np.linspace(-2e-3, 2e-3, 350)
intensity = (waist / radius)**2 * np.exp(-2 * (x[:, None] / radius)**2)
fig, ax = plt.subplots(figsize=(8, 4))
field = ax.pcolormesh(z, x * 1e3, intensity, shading="auto", cmap="viridis", vmin=0, vmax=1)
ax.plot(z, radius * 1e3, color="white", ls="--", lw=1.3, label="Beam radius ±w(z)")
ax.plot(z, -radius * 1e3, color="white", ls="--", lw=1.3)
ax.legend(loc="upper right")
ax.set(title="Gaussian beam through its waist", xlabel="Axial position (m)",
       ylabel="Transverse position (mm)")
fig.colorbar(field, ax=ax, label="Intensity relative to waist peak")
save_figure(fig, "gaussian_beam.png")

## Thin-lens focusing

The thin lens changes (q) through (1/q_{\rm out}=1/q_{\rm in}-1/f). The real part of this relation gives the change in wavefront curvature. The imaginary part of (1/q) gives the beam radius, so the radius is continuous across the lens. After the lens, propagation over distance (d) gives (q=q_{\rm out}+d), and the new waist occurs where its real part vanishes.

In [ ]:
q_in = lens_position + 1j * rayleigh
q_out = 1 / (1 / q_in - 1 / focal_length)
focus_distance = -q_out.real
focused_waist = beam_radius(1j * q_out.imag)

np.testing.assert_allclose(beam_radius(q_in), beam_radius(q_out))
assert focus_distance > 0 and focused_waist < waist

z = np.linspace(0, 0.80, 600)
q_free = z + 1j * rayleigh
q_lens = np.where(z < lens_position, q_free, q_out + z - lens_position)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(z, beam_radius(q_free) * 1e3, "--", color="#697887", label="Free space")
ax.plot(z, beam_radius(q_lens) * 1e3, color="#146b81", lw=2, label="With thin lens")
ax.axvline(lens_position, color="#9b5729", ls=":", label="Lens")
ax.scatter(lens_position + focus_distance, focused_waist * 1e3, color="#17283b",
           zorder=4, label="New waist")
ax.set(title="Beam radius with and without a thin lens", xlabel="Axial position (m)",
       ylabel="Beam radius (mm)")
ax.legend()
save_figure(fig, "thin_lens_focusing.png")
print(f"New waist: {focused_waist * 1e3:.3f} mm, {focus_distance:.3f} m after the lens")

## Wave plates and the polarization qubit

The Jones basis is ((|H\rangle, |V\rangle)). A half-wave plate at (45^\circ) has the Pauli-X Jones matrix in the chosen convention. A quarter-wave plate at (45^\circ) sends horizontal input to equal-amplitude components in quadrature. The probability bars show what an H/V measurement sees. The transverse field trajectories distinguish linear and circular polarization.

In [ ]:
def half_wave_plate(theta):
    c, s = np.cos(2 * theta), np.sin(2 * theta)
    return np.array([[c, s], [s, -c]])


def quarter_wave_plate(theta):
    c, s = np.cos(theta), np.sin(theta)
    cross = (1 - 1j) * s * c
    return np.array([[c**2 + 1j * s**2, cross],
                     [cross, s**2 + 1j * c**2]])


horizontal = np.array([1, 0], dtype=complex)
vertical = np.array([0, 1], dtype=complex)
plus = (horizontal + vertical) / np.sqrt(2)
pauli_x = np.array([[0, 1], [1, 0]])
hwp = half_wave_plate(np.pi / 4)
qwp = quarter_wave_plate(np.pi / 4)
hwp_output = hwp @ horizontal
qwp_output = qwp @ horizontal

for matrix in (hwp, qwp):
    np.testing.assert_allclose(matrix.conj().T @ matrix, np.eye(2), atol=1e-12)
np.testing.assert_allclose(hwp, pauli_x, atol=1e-12)
np.testing.assert_allclose(hwp_output, vertical, atol=1e-12)
np.testing.assert_allclose(pauli_x @ plus, plus, atol=1e-12)
np.testing.assert_allclose(abs(qwp_output)**2, [0.5, 0.5], atol=1e-12)
np.testing.assert_allclose(qwp_output[1] / qwp_output[0], -1j, atol=1e-12)

phase = np.linspace(0, 2 * np.pi, 400)
fig, axes = plt.subplots(2, 3, figsize=(9, 5.6), gridspec_kw={"height_ratios": [2, 1]})
for i, (title, state) in enumerate([
    ("Horizontal input", horizontal),
    ("Half-wave plate at 45°", hwp_output),
    ("Quarter-wave plate at 45°", qwp_output),
]):
    trajectory = np.real(state[:, None] * np.exp(-1j * phase))
    ax = axes[0, i]
    ax.plot(*trajectory, color="#146b81", lw=2)
    ax.axhline(0, lw=0.8, color="#b4c0ca", zorder=0)
    ax.axvline(0, lw=0.8, color="#b4c0ca", zorder=0)
    ax.set(xlim=(-1.05, 1.05), ylim=(-1.05, 1.05), aspect="equal",
           xlabel="Horizontal field", ylabel="Vertical field", title=title)
    bars = axes[1, i]
    bars.bar(["H", "V"], abs(state)**2, color=["#146b81", "#af642f"], width=0.52)
    bars.set(ylim=(0, 1.08), ylabel="Probability" if i == 0 else "")
    bars.set_yticks([0, 0.5, 1])
save_figure(fig, "polarization_gates.png")
print("Verified: X|H⟩ = |V⟩, X|+⟩ = |+⟩, and the quarter-wave plate adds a relative phase.")